# Classical (ExoGibbs) vs. Transformer Emulator — replacement test

Companion to `equilibrium_chemistry_transformer.ipynb`. The goal here is to
actually stress the pieces a production retrieval would hit:

- **8 retrieval parameters** — `T0, alpha, logg` plus the five log elemental
  abundances `log_He_H, log_C_H, log_O_H, log_N_H, log_S_H` sampled independently.
  The emulator's whole point is the 5-element basis; a 1-D `logZ` collapses that.
- **CO + H2O + H2–H2 CIA** — adding H2O means the emulator's H2O predictions
  actually enter the likelihood. With only CO, 15 of the emulator's 17 species
  never got tested.
- **Traced mean molecular weight** — computed from the full 17-species VMR
  profile. With `mmw` fixed at 2.33 the abundance parameters don't enter the
  continuum, which artificially flattens the likelihood.
- **Two ground truths** — (a) **solar composition** (C/O = 0.588) and
  (b) **C/O = 0.8** (C/H elevated at solar O/H). Everything — MAPs, MCMCs,
  summary tables — is run for both so systematic bias from the emulator is
  visible rather than hidden behind a single truth.

### What gets compared

1. Forward chemistry timing (hot JIT) for the two backends.
2. Forward spectrum timing for the two backends.
3. **4 MAP retrievals** — `{solar, CO08} × {classical, emulator}` — fast, deterministic.
4. **4 longer NUTS MCMC runs** — same 4 combinations, 100 warmup + 150 samples
   per run, `max_tree_depth=7`, 8 parameters, 1 chain per run.
5. Corner plots per truth, overlaying both chains. Includes derived **C/O ratio**
   posterior since that's the single parameter that differs between the two truths.

### Sizing knobs for a ~1-hour budget

| knob | main retrieval | this notebook |
|---|---|---|
| wavenumber points | 1050 | **400** |
| atmospheric layers | 50 | **40** (emulator min) |
| opacity sources | CO + CIA | **CO + H2O + CIA** |
| mmw | 2.33 (const) | **traced from VMR** |
| free parameters | 6 + sigma | **8** (T0, α, logg, 5×log_X_H) |
| MCMC warmup / samples | 200 / 300 | **100 / 150** per run, 4 runs |
| `max_tree_depth` | 11 | **7** |
| abundance priors | wide | **Normal(solar, 0.4 dex)** |
| truths tested | 1 (solar) | **2** (solar + C/O=0.8) |

Mock data is generated by the **classical** model at each truth, so the emulator
is measured by how close it gets to ExoGibbs-consistent ground truth. The
abundance priors are deliberately widened to 0.4 dex — narrow enough that the
sampler still moves in the hour budget, wide enough that weakly-constrained
parameters (He, N, S) expose posterior geometry issues if any are present.
Running classical and emulator under identical wide priors also doubles as a
free diagnostic: if the emulator stalls where classical does not, the
emulator's gradient field is the cause, not the problem geometry.

In [ ]:
from jax import config
config.update("jax_enable_x64", True)

import sys, types, time
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from numpy.random import default_rng

print("jax backend :", jax.default_backend(), "| devices:", jax.devices())

BUNDLE_PATH = Path("../models/fastchem_analytic_500k/best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

# The bundle is weights + metadata; the forward pass lives in
# src.models.transformer. The distribution keeps ``src/`` next to
# ``models/`` so BUNDLE_PATH.parents[2] is the distribution root — put it
# on sys.path to expose the inference code that shipped with the weights.
DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
sys.path.insert(0, str(DIST_ROOT))


## 1. Reduced ExoJAX forward model

A narrow wavenumber window that still straddles the CO band head plus H2O rovib
lines, on 40 atmospheric layers (the emulator's lower limit). Cross-section
matrices and CIA both scale with `N_layer × N_nu`, so the shorter grid is the
main source of speedup over the main retrieval notebook. Opacity sources below
are **CO + H2O + H2–H2 CIA**, with `mmw` computed per layer from the full
17-species VMR profile.


In [ ]:
from exojax.utils.grids import wavenumber_grid

# Slightly narrower window than the main notebook (22920-23000 AA, 1050 pts).
# Covers the CO band head around 22940 AA / 4360 cm^-1.
nu_grid, wav, resolution = wavenumber_grid(
    22930.0, 22990.0, 400, unit="AA", xsmode="premodit"
)
print("Resolution =", resolution, "  N(nu) =", len(nu_grid))

In [ ]:
from exojax.database.exomol.api import MdbExomol
from exojax.database.hitran.api import MdbHitran
from exojax.opacity import OpaPremodit

# CO from ExoMol (main line opacity in this band).
mdb = MdbExomol(".database/CO/12C-16O/Li2015", nurange=nu_grid)
opa = OpaPremodit(mdb, nu_grid, auto_trange=[500.0, 1500.0],
                  dit_grid_resolution=1.0, allow_32bit=True)

# H2O from HITRAN (auto-downloads par file on first use).
mdb_h2o = MdbHitran("H2O", nurange=nu_grid)
opa_h2o = OpaPremodit(mdb_h2o, nu_grid, auto_trange=[500.0, 1500.0],
                      dit_grid_resolution=1.0, allow_32bit=True)
MOLMASS_H2O = float(mdb_h2o.molmass)
print(f"H2O lines in band: {len(mdb_h2o.nu_lines)}   molmass: {MOLMASS_H2O:.4f}")

In [ ]:
from exojax.rt import ArtEmisPure
from exojax.utils.astrofunc import gravity_jupiter
from exojax.utils.zsol import nsol as _nsol

NLAYER = 40  # emulator requires num_levels in [40, 60]; pick the min for speed.
art = ArtEmisPure(
    nu_grid=nu_grid,
    pressure_btm=1.0e1,
    pressure_top=1.0e-5,
    nlayer=NLAYER,
    rtsolver="ibased",
    nstream=8,
)
art.change_temperature_range(500.0, 1500.0)

# Shared T-P and geometry truth values (same as main retrieval notebook).
LOGG_TRUTH = float(np.log10(gravity_jupiter(1.0, 10.0)))
RV_TRUTH, VSINI_TRUTH = 40.0, 10.0

# Asplund 2021 solar abundances, used as the reference for both truths.
_solar = _nsol()  # re-used when load-exogibbs runs; defined here for TRUTH dicts.
LOG_SOLAR = {f"log_{k}_H": float(np.log10(_solar[k])) for k in ("He", "C", "O", "N", "S")}

def _make_truth(name, overrides=None):
    d = dict(T0=1500.0, alpha=0.10, logg=LOGG_TRUTH,
             RV=RV_TRUTH, vsini=VSINI_TRUTH, **LOG_SOLAR)
    if overrides:
        d.update(overrides)
    d["_name"] = name
    return d

# Two ground truths — solar and C/O = 0.8 (C/H elevated at solar O/H).
# Solar C/O = 10**(log_C_H - log_O_H) ≈ 0.589. Target 0.8 → Δ log_C_H = +0.132 dex.
TRUTH_SOLAR = _make_truth("solar")
_co_shift = float(np.log10(0.8) - (LOG_SOLAR["log_C_H"] - LOG_SOLAR["log_O_H"]))
TRUTH_CO08 = _make_truth("CO08",
                         overrides={"log_C_H": LOG_SOLAR["log_C_H"] + _co_shift})

TRUTHS = [TRUTH_SOLAR, TRUTH_CO08]

for T in TRUTHS:
    co = 10.0 ** (T["log_C_H"] - T["log_O_H"])
    print(f"TRUTH[{T['_name']:<6}]  C/O = {co:.3f}  "
          f"log_C_H = {T['log_C_H']:+.4f}  log_O_H = {T['log_O_H']:+.4f}")

Tarr_truth = art.powerlaw_temperature(TRUTH_SOLAR["T0"], TRUTH_SOLAR["alpha"])
print("Tarr_truth[0],[-1] :", float(Tarr_truth[0]), float(Tarr_truth[-1]))

In [ ]:
from exojax.database.contdb import CdbCIA
from exojax.opacity import OpaCIA
from exojax.postproc.specop import SopRotation, SopInstProfile
from exojax.utils.instfunc import resolution_to_gaussian_std

cdb = CdbCIA(".database/H2-H2_2011.cia", nurange=nu_grid)
opacia = OpaCIA(cdb, nu_grid=nu_grid)
sop_rot = SopRotation(nu_grid, vsini_max=100.0)
sop_inst = SopInstProfile(nu_grid, vrmax=1000.0)

# Observing / instrumental configuration (same numbers as the main notebook).
RES_INST = 70000.0
BETA_INST = resolution_to_gaussian_std(RES_INST)
U1, U2 = 0.0, 0.0
# mmw is no longer a constant here — it is computed per layer from the 17-species
# emulator VMR in fspec-defs, which makes the continuum respond to the abundance
# parameters (the whole point of a real replacement test).

# Observation grid: every 5th point of nu_grid, drop the last 20 to avoid edge artifacts.
nu_obs = np.asarray(nu_grid[::5][:-20])
print("N(nu_obs) =", len(nu_obs))

## 2. Two chemistry backends

Both backends take a 5-D element vector and return a VMR table of shape
`(NLAYER, 17)` — the emulator directly, ExoGibbs via the `EG_IDX_17` gather
mapping. The emulator exposes 17 species against a 5-element basis
`(He, C, O, N, S)`; ExoGibbs exposes 523 species against a 28-element AAG21
basis (all non-{H,He,C,O,N,S} elements are held fixed at AAG21 solar on the
ExoGibbs side, so the two are comparable on this basis).

In [ ]:
from src.models.standalone_inference import load_model, make_fastchem_vmr_fn

bundle = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")

print("chemistry   :", bundle.chemistry_type, " | model:", bundle.model_type)
print("species (17):", species_labels)
print("global order:", bundle.data_contract["global_static_feature_order"])
print("fixed_globals (must be {}):", bundle.fixed_globals)

# Molar masses for the 17 emulator species (g/mol), matched positionally to species_labels.
# Used to compute mmw per layer from the VMR profile.
MOLAR_MASS = {"H2": 2.016, "He": 4.003, "H": 1.008, "O": 15.999, "OH": 17.007,
              "H2O": 18.015, "CO": 28.010, "CO2": 44.009, "CH4": 16.043, "N2": 28.014,
              "NH3": 17.031, "H2S": 34.081, "SH": 33.073, "S": 32.065, "SO": 48.064,
              "SO2": 64.064, "S2": 64.130}
MASS_VEC_17 = jnp.array([MOLAR_MASS[s] for s in species_labels])

IDX_CO_ML  = species_labels.index("CO")
IDX_H2_ML  = species_labels.index("H2")
IDX_H2O_ML = species_labels.index("H2O")

In [ ]:
from exogibbs.presets.fastchem import chemsetup
from exogibbs.api import get_default_equilibrium_grid_path, load_equilibrium_grid_netcdf
from exogibbs.api.equilibrium import (
    EquilibriumOptions, GridEquilibriumInitializer, equilibrium_profile,
)

chem = chemsetup()
grid_path = get_default_equilibrium_grid_path("fastchem")
grid = load_equilibrium_grid_netcdf(str(grid_path))
initializer = GridEquilibriumInitializer(grid=grid, preset_name="fastchem")
opts = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")

# Element indices (into chem.elements) that the emulator treats as free.
idx_He_elem = chem.elements.index("He")
idx_C_elem  = chem.elements.index("C")
idx_O_elem  = chem.elements.index("O")
idx_N_elem  = chem.elements.index("N")
idx_S_elem  = chem.elements.index("S")
ELEM_IDX_5 = jnp.array([idx_He_elem, idx_C_elem, idx_O_elem, idx_N_elem, idx_S_elem])

# ExoGibbs species names that correspond to the emulator's 17 species, in the
# same positional order as species_labels. Used to extract the matching subset
# from ExoGibbs VMRs so the two backends produce (NLAYER, 17) tables.
ML_TO_EG_NAME = {
    "H2": "H2", "He": "He1", "H": "H1", "O": "O1", "OH": "H1O1",
    "H2O": "H2O1", "CO": "C1O1", "CO2": "C1O2", "CH4": "C1H4",
    "N2": "N2", "NH3": "H3N1", "H2S": "H2S1", "SH": "H1S1",
    "S": "S1", "SO": "O1S1", "SO2": "O2S1", "S2": "S2",
}
EG_IDX_17 = jnp.array([chem.species.index(ML_TO_EG_NAME[s]) for s in species_labels])
IDX_CO_EG  = chem.species.index("C1O1")
IDX_H2_EG  = chem.species.index("H2")
IDX_H2O_EG = chem.species.index("H2O1")

# Full AAG21 element vector for ExoGibbs (28 entries incl. trailing e- = 0).
element_vector_solar = jnp.append(
    jnp.array([_solar[el] for el in chem.elements[:-1]]), 0.0
)

# Matching 5-element global_inputs for the emulator (solar).
ML_SOLAR = {k: float(_solar[k.split("_")[0]]) for k in ("He_H", "C_H", "O_H", "N_H", "S_H")}
print("ExoGibbs elements   :", len(chem.elements), " | species:", len(chem.species))
print("ML solar abundances :", ML_SOLAR)
print("ELEM_IDX_5          :", np.asarray(ELEM_IDX_5))
print("EG_IDX_17 (sample)  :", {s: int(chem.species.index(ML_TO_EG_NAME[s])) for s in species_labels[:6]})

## 3. Forward chemistry timing

One-shot and JIT-hot timings for a single `(Tarr, pressure, 5-abundances)` call.
JIT-hot is what the MCMC and MAP loops actually see. Both backends return a
`(NLAYER, 17)` VMR table aligned to `species_labels`.

In [ ]:
ELEM_NAMES = ("He", "C", "O", "N", "S")
ABUND_PARAMS = tuple(f"log_{e}_H" for e in ELEM_NAMES)  # ordering fixed across the notebook.

def _element_vector(log_abund):
    """Overwrite the He/C/O/N/S entries of the AAG21 vector with 10**log_abund."""
    lin = 10.0 ** log_abund                       # (5,)
    return element_vector_solar.at[ELEM_IDX_5].set(lin)

def _ml_globals(log_abund):
    lin = 10.0 ** log_abund
    return {f"{e}_H": lin[i] for i, e in enumerate(ELEM_NAMES)}

@jax.jit
def vmr_classical(Tarr, log_abund):
    """ExoGibbs -> (NLAYER, 17) VMR table aligned to species_labels."""
    ev = _element_vector(log_abund)
    res = equilibrium_profile(chem, Tarr, art.pressure, ev, Pref=1.0,
                              initializer=initializer, options=opts)
    return res.x[:, EG_IDX_17]

@jax.jit
def vmr_emulator(Tarr, log_abund):
    """Emulator -> (NLAYER, 17) VMR table (already aligned)."""
    return vmr_fn(Tarr, art.pressure, _ml_globals(log_abund))

ABUND_SOLAR_VEC = jnp.array([LOG_SOLAR[p] for p in ABUND_PARAMS])

# Warmup (forces compilation).
_ = vmr_classical(Tarr_truth, ABUND_SOLAR_VEC).block_until_ready()
_ = vmr_emulator(Tarr_truth, ABUND_SOLAR_VEC).block_until_ready()

def _bench(fn, *args, n=20):
    fn(*args).block_until_ready()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        ts.append(time.perf_counter() - t0)
    return np.median(ts) * 1e3

t_cl = _bench(vmr_classical, Tarr_truth, ABUND_SOLAR_VEC)
t_ml = _bench(vmr_emulator,  Tarr_truth, ABUND_SOLAR_VEC)
print(f"vmr_classical (ExoGibbs)   JIT-hot : {t_cl:7.2f} ms")
print(f"vmr_emulator  (transformer) JIT-hot : {t_ml:7.2f} ms  "
      f"(x{t_cl/max(t_ml,1e-6):.2f} vs ExoGibbs)")

# Agreement plot on four key species (same four the retrieval actually depends on).
vc = np.asarray(vmr_classical(Tarr_truth, ABUND_SOLAR_VEC))
vm = np.asarray(vmr_emulator(Tarr_truth, ABUND_SOLAR_VEC))
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, sp in zip(axes, ("H2", "CO", "H2O", "CH4")):
    i = species_labels.index(sp)
    ax.plot(vc[:, i], art.pressure, label="ExoGibbs")
    ax.plot(vm[:, i], art.pressure, ls="--", label="emulator")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.invert_yaxis()
    ax.set_xlabel(f"VMR ({sp})"); ax.set_title(sp)
    ax.legend(fontsize=9)
axes[0].set_ylabel("Pressure (bar)")
plt.tight_layout(); plt.show()

## 4. Full forward spectrum models

Two `fspec(...)` functions that differ **only** in the chemistry line. Both take
8 parameters `(T0, alpha, logg, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)`.
Both compute `mmw` per layer from the full 17-species VMR table. Both add CO +
H2O line opacity and H2–H2 CIA. The `_fspec_core` routine handles the identical
downstream radiative-transfer / rotation / instrumental / resampling stack, so
any retrieval difference downstream is attributable to the chemistry backend.

In [ ]:
from exojax.atm.atmconvert import vmr_to_mmr
from exojax.database.molinfo.mass import isotope_molmass

MOLMASS_CO = isotope_molmass("12C-16O")

# Parameter ordering shared with MAP / MCMC cells.
PARAMS = ("T0", "alpha", "logg") + ABUND_PARAMS
PARAM_IDX = {p: i for i, p in enumerate(PARAMS)}

def _fspec_core(Tarr, gravity, RV, vsini, vmr_table):
    """Identical downstream RT used by both backends. vmr_table: (NLAYER, 17)."""
    vmr_co  = vmr_table[:, IDX_CO_ML]
    vmr_h2o = vmr_table[:, IDX_H2O_ML]
    vmr_h2  = vmr_table[:, IDX_H2_ML]
    # Per-layer mean molecular weight traced from the full VMR table.
    mmw = jnp.sum(vmr_table * MASS_VEC_17, axis=-1)            # (NLAYER,)

    mmr_co  = vmr_to_mmr(vmr_co,  MOLMASS_CO,  mmw)
    mmr_h2o = vmr_to_mmr(vmr_h2o, MOLMASS_H2O, mmw)

    xs_co  = opa.xsmatrix(Tarr, art.pressure)
    xs_h2o = opa_h2o.xsmatrix(Tarr, art.pressure)
    dtau_co  = art.opacity_profile_xs(xs_co,  mmr_co,  MOLMASS_CO,  gravity)
    dtau_h2o = art.opacity_profile_xs(xs_h2o, mmr_h2o, MOLMASS_H2O, gravity)

    logacia = opacia.logacia_matrix(Tarr)
    # opacity_profile_cia expects mmw broadcastable against (N_layer, N_nu);
    # a (L, 1) shape is the right way to pass a per-layer mmw.
    dtau_cia = art.opacity_profile_cia(logacia, Tarr, vmr_h2, vmr_h2,
                                       mmw[:, None], gravity)

    F = art.run(dtau_co + dtau_h2o + dtau_cia, Tarr)
    Frot = sop_rot.rigid_rotation(F, vsini, U1, U2)
    Finst = sop_inst.ipgauss(Frot, BETA_INST)
    return sop_inst.sampling(Finst, RV, nu_obs)

def _build_fspec(vmr_backend):
    def fspec(T0, alpha, logg, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H,
              RV=RV_TRUTH, vsini=VSINI_TRUTH):
        Tarr = art.powerlaw_temperature(T0, alpha)
        log_abund = jnp.array([log_He_H, log_C_H, log_O_H, log_N_H, log_S_H])
        vmr = vmr_backend(Tarr, log_abund)
        return _fspec_core(Tarr, 10.0 ** logg, RV, vsini, vmr)
    return fspec

fspec_classical = _build_fspec(vmr_classical)
fspec_emulator  = _build_fspec(vmr_emulator)
fspec_classical_jit = jax.jit(fspec_classical)
fspec_emulator_jit  = jax.jit(fspec_emulator)

def _kw(T):
    return {p: T[p] for p in PARAMS}

# JIT warmup (compilation is the one-time cost).
t0 = time.perf_counter()
_ = fspec_classical_jit(**_kw(TRUTH_SOLAR)).block_until_ready()
print(f"classical first-call (compile + run) : {time.perf_counter()-t0:.2f} s")

t0 = time.perf_counter()
_ = fspec_emulator_jit(**_kw(TRUTH_SOLAR)).block_until_ready()
print(f"emulator  first-call (compile + run) : {time.perf_counter()-t0:.2f} s")

def _bench_kw(fn, n=15, **kw):
    fn(**kw).block_until_ready()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn(**kw).block_until_ready()
        ts.append(time.perf_counter() - t0)
    return np.median(ts) * 1e3

t_fsc = _bench_kw(fspec_classical_jit, **_kw(TRUTH_SOLAR))
t_fsm = _bench_kw(fspec_emulator_jit,  **_kw(TRUTH_SOLAR))
print(f"fspec_classical JIT-hot : {t_fsc:7.2f} ms / call")
print(f"fspec_emulator  JIT-hot : {t_fsm:7.2f} ms / call  "
      f"(x{t_fsc/max(t_fsm,1e-6):.2f} vs classical)")

FSPEC_TIMING = {"fspec_classical_ms": float(t_fsc),
                "fspec_emulator_ms": float(t_fsm)}

## 5. Mock spectra (classical ground truth × 2)

Two mock spectra — one at **solar** (C/O = 0.588) and one at **C/O = 0.8** —
generated by the **classical** forward model at the truth parameters plus
Gaussian noise. ExoGibbs sees its own model in the data; the emulator has to
approximate it. Any bias in the emulator's posterior will show up as an offset
from the truth values, per truth.

In [ ]:
NOISE = 500.0
rng = default_rng(0)

MOCKS = {}
for T in TRUTHS:
    mu = fspec_classical_jit(**_kw(T))
    Fobs = np.asarray(mu) + rng.normal(0.0, NOISE, size=len(nu_obs))
    MOCKS[T["_name"]] = {"truth": T, "mu": np.asarray(mu), "Fobs": Fobs}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for ax, T in zip(axes, TRUTHS):
    M = MOCKS[T["_name"]]
    mu_ml = np.asarray(fspec_emulator_jit(**_kw(T)))
    ax.plot(nu_obs, M["mu"], label="classical truth (noiseless)", lw=1.2)
    ax.plot(nu_obs, mu_ml, ls="--", lw=1.0, label="emulator @ truth")
    ax.errorbar(nu_obs, M["Fobs"], NOISE, fmt=".", color="gray",
                alpha=0.4, label="mock data")
    co = 10.0 ** (T["log_C_H"] - T["log_O_H"])
    ax.set_title(f"{T['_name']}: C/O = {co:.3f}  (N={len(nu_obs)}, noise={NOISE:g})")
    ax.set_xlabel("wavenumber (cm$^{-1}$)")
    ax.legend(fontsize=9)
axes[0].set_ylabel("flux (erg/s/cm$^2$/cm$^{-1}$)")
plt.tight_layout(); plt.show()

## 6. MAP retrievals — 4 runs

L-BFGS-B on the negative log-likelihood (flat priors treated as box bounds).
Four runs: `{solar, CO08} × {classical, emulator}`. Same `X0` (offset from
both truths on purpose so the optimizer actually does work) and same bounds
for every run, so retrieval wall-clock and final `nll` are directly
comparable.

In [ ]:
from scipy.optimize import minimize

# Bounds per parameter. Widened to match the wider NUTS priors below:
# abundance bounds are solar ± 1.2 dex (≈ ±3σ of a 0.4-dex Gaussian prior).
BOUNDS_DICT = {
    "T0":       (1200.0, 1800.0),
    "alpha":    (0.03, 0.22),
    "logg":     (3.8, 5.2),
    "log_He_H": (LOG_SOLAR["log_He_H"] - 0.8, LOG_SOLAR["log_He_H"] + 0.8),
    "log_C_H":  (LOG_SOLAR["log_C_H"]  - 1.2, LOG_SOLAR["log_C_H"]  + 1.2),
    "log_O_H":  (LOG_SOLAR["log_O_H"]  - 1.2, LOG_SOLAR["log_O_H"]  + 1.2),
    "log_N_H":  (LOG_SOLAR["log_N_H"]  - 1.2, LOG_SOLAR["log_N_H"]  + 1.2),
    "log_S_H":  (LOG_SOLAR["log_S_H"]  - 1.2, LOG_SOLAR["log_S_H"]  + 1.2),
}
BOUNDS = [BOUNDS_DICT[p] for p in PARAMS]

# Start above all truths so the optimizer has real work to do for both.
X0_DICT = {
    "T0": 1400.0, "alpha": 0.08, "logg": 4.2,
    "log_He_H": LOG_SOLAR["log_He_H"] + 0.10,
    "log_C_H":  LOG_SOLAR["log_C_H"]  + 0.25,
    "log_O_H":  LOG_SOLAR["log_O_H"]  + 0.10,
    "log_N_H":  LOG_SOLAR["log_N_H"]  + 0.10,
    "log_S_H":  LOG_SOLAR["log_S_H"]  + 0.10,
}
X0 = np.array([X0_DICT[p] for p in PARAMS])

def _nll_factory(fspec_jit, Fobs_j):
    @jax.jit
    def nll(x):
        mu = fspec_jit(*[x[i] for i in range(len(PARAMS))])
        return 0.5 * jnp.sum(((mu - Fobs_j) / NOISE) ** 2)
    return nll, jax.jit(jax.grad(nll))

NLLS = {}
for T in TRUTHS:
    name = T["_name"]
    Fobs_j = jnp.asarray(MOCKS[name]["Fobs"])
    nll_cl, gnll_cl = _nll_factory(fspec_classical_jit, Fobs_j)
    nll_ml, gnll_ml = _nll_factory(fspec_emulator_jit,  Fobs_j)
    # Warmup (cache tracing).
    for fn in (nll_cl, gnll_cl, nll_ml, gnll_ml):
        _ = fn(jnp.asarray(X0)).block_until_ready()
    NLLS[name] = dict(classical=(nll_cl, gnll_cl), emulator=(nll_ml, gnll_ml),
                      Fobs_j=Fobs_j)
    x_truth = jnp.asarray([T[p] for p in PARAMS])
    print(f"[{name:<6}] nll @ truth  cl={float(nll_cl(x_truth)):.3f}  "
          f"ml={float(nll_ml(x_truth)):.3f}   "
          f"nll @ X0  cl={float(nll_cl(jnp.asarray(X0))):.1f}  "
          f"ml={float(nll_ml(jnp.asarray(X0))):.1f}")

In [ ]:
def run_map(nll, gnll, label):
    obj = lambda x: float(nll(jnp.asarray(x)))
    jac = lambda x: np.asarray(gnll(jnp.asarray(x)), dtype=np.float64)
    t0 = time.perf_counter()
    res = minimize(obj, X0, jac=jac, method="L-BFGS-B", bounds=BOUNDS,
                   options=dict(maxiter=200, ftol=1e-10, gtol=1e-8))
    dt = time.perf_counter() - t0
    print(f"[{label:<16}] nfev={res.nfev:3d} njev={res.njev:3d} "
          f"success={res.success} nll*={res.fun:.3f}  wall={dt:.2f} s")
    return dict(x=res.x, nll=float(res.fun), nfev=int(res.nfev),
                njev=int(res.njev), wall_s=dt, success=bool(res.success))

MAPS = {}
for T in TRUTHS:
    name = T["_name"]
    nll_cl, gnll_cl = NLLS[name]["classical"]
    nll_ml, gnll_ml = NLLS[name]["emulator"]
    MAPS[name] = {
        "classical": run_map(nll_cl, gnll_cl, f"{name}/classical"),
        "emulator":  run_map(nll_ml, gnll_ml, f"{name}/emulator "),
    }

# Retrieved values vs truth, side-by-side per truth.
print()
for T in TRUTHS:
    name = T["_name"]
    hdr = f"  {'param':<10}{'truth':>10}{'classical MAP':>16}{'emulator MAP':>16}{'Δ(ML-truth)':>16}"
    print(f"[{name}]"); print(hdr); print("  " + "-" * (len(hdr) - 2))
    for i, p in enumerate(PARAMS):
        t = T[p]
        c = MAPS[name]["classical"]["x"][i]
        m = MAPS[name]["emulator"]["x"][i]
        print(f"  {p:<10}{t:>10.4f}{c:>16.4f}{m:>16.4f}{m-t:>16.4f}")
    print()

In [ ]:
# Overlay MAP fits on each mock spectrum.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for ax, T in zip(axes, TRUTHS):
    name = T["_name"]
    M = MOCKS[name]
    mu_cl = np.asarray(fspec_classical_jit(*MAPS[name]["classical"]["x"]))
    mu_ml = np.asarray(fspec_emulator_jit(*MAPS[name]["emulator"]["x"]))
    ax.errorbar(nu_obs, M["Fobs"], NOISE, fmt=".", color="gray",
                alpha=0.35, label="mock data")
    ax.plot(nu_obs, mu_cl, lw=1.1,
            label=f"classical MAP ({MAPS[name]['classical']['wall_s']:.1f}s)")
    ax.plot(nu_obs, mu_ml, lw=1.1, ls="--",
            label=f"emulator MAP ({MAPS[name]['emulator']['wall_s']:.1f}s)")
    ax.set_title(f"{name}  —  MAP fits")
    ax.set_xlabel("wavenumber (cm$^{-1}$)")
    ax.legend(fontsize=9)
axes[0].set_ylabel("flux (erg/s/cm$^2$/cm$^{-1}$)")
plt.tight_layout(); plt.show()

## 7. Longer NUTS MCMC — 4 runs

One run per backend × truth: **100 warmup + 150 samples**, `max_tree_depth=7`,
`target_accept_prob=0.8`, single chain per run. Priors are deliberately widened
relative to the previous version — **Normal(solar, 0.4 dex)** on each abundance
— so weakly-constrained parameters (He, N, S) have room to reveal any posterior
geometry problems or emulator extrapolation issues.

Running classical and emulator under identical wide priors also gives you the
diagnostic from the discussion above for free: if the emulator stalls at a
tiny step size and the classical backend does not, the emulator's gradient
field is the cause rather than the posterior geometry.

In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random

NUM_WARMUP, NUM_SAMPLES = 100, 150
MAX_TREE = 7
ABUND_PRIOR_SIGMA = 0.4  # dex, centered on solar (widened from 0.15).

def _make_model(fspec_jit):
    def model(spectrum):
        T0    = numpyro.sample("T0",    dist.Uniform(1200.0, 1800.0))
        alpha = numpyro.sample("alpha", dist.Uniform(0.03, 0.22))
        logg  = numpyro.sample("logg",  dist.Uniform(3.8, 5.2))
        log_He = numpyro.sample("log_He_H",
                                dist.Normal(LOG_SOLAR["log_He_H"], ABUND_PRIOR_SIGMA))
        log_C  = numpyro.sample("log_C_H",
                                dist.Normal(LOG_SOLAR["log_C_H"],  ABUND_PRIOR_SIGMA))
        log_O  = numpyro.sample("log_O_H",
                                dist.Normal(LOG_SOLAR["log_O_H"],  ABUND_PRIOR_SIGMA))
        log_N  = numpyro.sample("log_N_H",
                                dist.Normal(LOG_SOLAR["log_N_H"],  ABUND_PRIOR_SIGMA))
        log_S  = numpyro.sample("log_S_H",
                                dist.Normal(LOG_SOLAR["log_S_H"],  ABUND_PRIOR_SIGMA))
        mu = fspec_jit(T0, alpha, logg, log_He, log_C, log_O, log_N, log_S)
        numpyro.sample("spectrum", dist.Normal(mu, NOISE), obs=spectrum)
    return model

def run_mcmc(fspec_jit, Fobs, label, seed=0):
    model = _make_model(fspec_jit)
    kernel = NUTS(model, forward_mode_differentiation=False,
                  max_tree_depth=MAX_TREE, target_accept_prob=0.8)
    mcmc = MCMC(kernel, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES, num_chains=1,
                progress_bar=True)
    t0 = time.perf_counter()
    mcmc.run(random.PRNGKey(seed), spectrum=jnp.asarray(Fobs),
             extra_fields=("num_steps", "diverging", "accept_prob",
                           "mean_accept_prob", "energy", "potential_energy"))
    dt = time.perf_counter() - t0
    n_iter = NUM_WARMUP + NUM_SAMPLES
    print(f"[{label:<18}] MCMC wall: {dt:6.1f} s  ({dt/n_iter*1e3:6.0f} ms/iter)")
    mcmc.print_summary()
    return mcmc, dt

In [ ]:
# Classical backend on both mocks (solar, then CO08).
MCMC_CL = {}
for T in TRUTHS:
    name = T["_name"]
    mcmc, dt = run_mcmc(fspec_classical_jit, MOCKS[name]["Fobs"],
                        label=f"{name}/classical")
    MCMC_CL[name] = {"mcmc": mcmc, "samples": mcmc.get_samples(), "wall_s": dt}

In [ ]:
# Emulator backend on both mocks (solar, then CO08).
MCMC_ML = {}
for T in TRUTHS:
    name = T["_name"]
    mcmc, dt = run_mcmc(fspec_emulator_jit, MOCKS[name]["Fobs"],
                        label=f"{name}/emulator")
    MCMC_ML[name] = {"mcmc": mcmc, "samples": mcmc.get_samples(), "wall_s": dt}

## 8. Summary — timings, corner plots, retrieved values

One corner plot per truth (solar, CO08), overlaying the two chains so bias
shows up as a split posterior cloud. A derived **C/O ratio** posterior follows
— that is the single parameter that separates the two truths, and the cleanest
thing to look at for replacement-test quality.

In [ ]:
from scipy.stats import gaussian_kde

def _corner(samples_list, labels, colors, names, truths=None, bounds=None,
            map_points=None, bins=25, figsize=(13, 13), title=None):
    """Corner plot for N chains (scipy-KDE only; no external corner library)."""
    n = len(labels)
    fig, axes = plt.subplots(n, n, figsize=figsize)
    arrs = [np.stack([np.asarray(s[p]) for p in labels], axis=1) for s in samples_list]

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False); continue
            if i == j:
                for arr, c in zip(arrs, colors):
                    x = arr[:, i]
                    ax.hist(x, bins=bins, density=True, color=c, alpha=0.25, edgecolor="none")
                    try:
                        kde = gaussian_kde(x)
                        xs = np.linspace(x.min(), x.max(), 200)
                        ax.plot(xs, kde(xs), color=c, lw=1.5)
                    except np.linalg.LinAlgError:
                        pass
                if truths is not None and labels[i] in truths:
                    ax.axvline(truths[labels[i]], color="k", ls=":", lw=1.2)
                if map_points is not None:
                    for mp, c in zip(map_points, colors):
                        ax.axvline(mp[i], color=c, ls="--", lw=1.0, alpha=0.9)
                ax.set_yticks([])
            else:
                for arr, c in zip(arrs, colors):
                    ax.scatter(arr[:, j], arr[:, i], s=5, color=c,
                               alpha=0.30, edgecolor="none")
                    try:
                        kde = gaussian_kde(np.vstack([arr[:, j], arr[:, i]]))
                        lo_j, hi_j = arr[:, j].min(), arr[:, j].max()
                        lo_i, hi_i = arr[:, i].min(), arr[:, i].max()
                        xx, yy = np.meshgrid(np.linspace(lo_j, hi_j, 50),
                                             np.linspace(lo_i, hi_i, 50))
                        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
                        ax.contour(xx, yy, zz, levels=3, colors=c,
                                   linewidths=0.8, alpha=0.85)
                    except (np.linalg.LinAlgError, ValueError):
                        pass
                if truths is not None:
                    if labels[j] in truths: ax.axvline(truths[labels[j]], color="k", ls=":", lw=0.9)
                    if labels[i] in truths: ax.axhline(truths[labels[i]], color="k", ls=":", lw=0.9)
                if map_points is not None:
                    for mp, c in zip(map_points, colors):
                        ax.plot(mp[j], mp[i], marker="s", color=c, ms=5,
                                mec="k", mew=0.5)
            if bounds is not None:
                ax.set_xlim(*bounds[j])
                if i != j: ax.set_ylim(*bounds[i])
            if i < n - 1: ax.set_xticklabels([])
            else:         ax.set_xlabel(labels[j], fontsize=9); ax.tick_params(axis="x", labelsize=7)
            if j > 0 or i == 0: ax.set_yticklabels([])
            else:               ax.set_ylabel(labels[i], fontsize=9); ax.tick_params(axis="y", labelsize=7)

    handles = [plt.Line2D([0], [0], color=c, lw=2, label=n_) for c, n_ in zip(colors, names)]
    if truths is not None:
        handles.append(plt.Line2D([0], [0], color="k", ls=":", lw=1.1, label="truth"))
    if map_points is not None:
        handles.append(plt.Line2D([0], [0], color="gray", ls="--", lw=1.0, label="MAP (diagonal)"))
        handles.append(plt.Line2D([0], [0], color="gray", marker="s", ms=5,
                                  ls="none", mec="k", mew=0.5, label="MAP (2D)"))
    axes[0, -1].set_visible(True); axes[0, -1].axis("off")
    axes[0, -1].legend(handles=handles, loc="center", fontsize=9, frameon=False)
    if title is not None:
        fig.suptitle(title, fontsize=11, y=0.995)
    fig.align_labels()
    plt.tight_layout()
    return fig, axes

for T in TRUTHS:
    name = T["_name"]
    co = 10.0 ** (T["log_C_H"] - T["log_O_H"])
    _ = _corner(
        [MCMC_CL[name]["samples"], MCMC_ML[name]["samples"]],
        labels=list(PARAMS),
        colors=["C0", "C1"],
        names=["classical (ExoGibbs)", "emulator (transformer)"],
        truths={p: T[p] for p in PARAMS},
        bounds=BOUNDS,
        map_points=[MAPS[name]["classical"]["x"], MAPS[name]["emulator"]["x"]],
        title=f"{name}  —  truth C/O = {co:.3f}",
    )
    plt.show()

# Derived C/O posterior per truth (cleanest single-number replacement test).
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for T in TRUTHS:
    name = T["_name"]
    co_truth = 10.0 ** (T["log_C_H"] - T["log_O_H"])
    for backend, store, style in [("classical", MCMC_CL, "-"),
                                  ("emulator",  MCMC_ML, "--")]:
        s = store[name]["samples"]
        co = 10.0 ** (np.asarray(s["log_C_H"]) - np.asarray(s["log_O_H"]))
        ax.hist(co, bins=18, density=True, alpha=0.35,
                histtype="stepfilled" if style == "-" else "step",
                linestyle=style, lw=1.8,
                label=f"{name} / {backend}")
    ax.axvline(co_truth, color="k", ls=":", lw=1.0)
ax.set_xlabel("C/O ratio (derived)"); ax.set_ylabel("density")
ax.set_title("Derived C/O posterior — dotted lines are truths (0.588, 0.800)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
def _summ(samples):
    return {p: (float(np.mean(samples[p])), float(np.std(samples[p]))) for p in PARAMS}

print("=" * 88)
print("TIMING SUMMARY")
print("=" * 88)
print(f"  fspec JIT-hot (ms/call)     classical: {FSPEC_TIMING['fspec_classical_ms']:7.2f}   "
      f"emulator: {FSPEC_TIMING['fspec_emulator_ms']:7.2f}   "
      f"(emulator is {FSPEC_TIMING['fspec_classical_ms']/max(FSPEC_TIMING['fspec_emulator_ms'],1e-6):.2f}x classical)")
for T in TRUTHS:
    name = T["_name"]
    print(f"  [{name}] MAP wall (s)        classical: {MAPS[name]['classical']['wall_s']:7.2f}   "
          f"emulator: {MAPS[name]['emulator']['wall_s']:7.2f}   "
          f"(nfev cl={MAPS[name]['classical']['nfev']}, ml={MAPS[name]['emulator']['nfev']})")
    print(f"  [{name}] NUTS wall (s,{NUM_WARMUP}+{NUM_SAMPLES}) classical: {MCMC_CL[name]['wall_s']:7.1f}   "
          f"emulator: {MCMC_ML[name]['wall_s']:7.1f}")
print()

for T in TRUTHS:
    name = T["_name"]
    S_CL = _summ(MCMC_CL[name]["samples"])
    S_ML = _summ(MCMC_ML[name]["samples"])
    co_truth = 10.0 ** (T["log_C_H"] - T["log_O_H"])

    print("=" * 88)
    print(f"RETRIEVED VALUES — {name}  (truth C/O = {co_truth:.3f})")
    print("=" * 88)
    hdr = (f"  {'param':<10}{'truth':>10}{'cl MAP':>12}{'cl MCMC':>20}"
           f"{'ml MAP':>12}{'ml MCMC':>20}")
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    for i, p in enumerate(PARAMS):
        t = T[p]
        mc, uc = S_CL[p]; mm, um = S_ML[p]
        print(f"  {p:<10}{t:>10.4f}{MAPS[name]['classical']['x'][i]:>12.4f}"
              f"{mc:>13.4f}±{uc:<5.3f}{MAPS[name]['emulator']['x'][i]:>12.4f}"
              f"{mm:>13.4f}±{um:<5.3f}")
    # Derived C/O summary.
    def _co(s):
        co = 10.0 ** (np.asarray(s["log_C_H"]) - np.asarray(s["log_O_H"]))
        return float(np.mean(co)), float(np.std(co))
    mc, uc = _co(MCMC_CL[name]["samples"])
    mm, um = _co(MCMC_ML[name]["samples"])
    print(f"  {'C/O':<10}{co_truth:>10.4f}{'':>12}{mc:>13.4f}±{uc:<5.3f}"
          f"{'':>12}{mm:>13.4f}±{um:<5.3f}")
    print()

print("Notes:")
print("  - Mock spectra are generated by the classical model, so systematic offsets")
print("    between emulator posterior means and the truth measure replacement-test bias.")
print("  - C/O is the cleanest single number: it separates the two truths and should")
print("    land on 0.588 (solar) and 0.800 (CO08) for both backends.")
print("  - NUTS is short (40+40, max_tree_depth=6) — rely on posterior means, not tails.")

## 9. Diagnostics dump — save everything for offline comparison

Writes one timestamped directory under `diagnostics/` per notebook run so
successive runs don't overwrite each other. Each run produces:

- `summary.json` — per-truth retrieval biases (MCMC mean − truth), posterior
  widths, classical/emulator posterior-width ratios, MAP/MCMC wall times,
  NLL at truth / at each MAP (cross-backend too), spectrum residual at truth
  (emulator − classical, noiseless), per-species VMR log10 ratio stats,
  NUTS divergence / step-count / accept-rate summary.
- `posteriors.npz` — raw NUTS samples keyed by `<truth>/<backend>/<param>`
  plus the derived `C_over_O` chain.
- `vmr_at_truth.npz` — `(NLAYER, 17)` VMR tables from both backends at each
  truth and the layerwise `log10(emulator/classical)` ratio.
- `spectra.npz` — mock `Fobs`, noiseless classical and emulator spectra at
  truth, and MAP-reconstructed spectra from each backend.
- `nuts_extras.npz` — per-iter NUTS diagnostics (num_steps, diverging,
  accept_prob, energy, potential_energy).

Headline print at the end is the quickest read: per-parameter bias and
posterior-width ratio emulator/classical. Anything with `|bias| > σ_cl` or
`width_ratio` far from 1 is where the emulator is disagreeing with the
classical backend.


In [ ]:
# Diagnostics dump — per-backend, per-truth snapshot for offline comparison.
import json, datetime
from pathlib import Path

OUTDIR = Path("diagnostics") / (
    "comparison_"
    + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
OUTDIR.mkdir(parents=True, exist_ok=True)
print("writing diagnostics to:", OUTDIR.resolve())

def _pstats(x):
    x = np.asarray(x)
    q = np.quantile(x, [0.025, 0.16, 0.5, 0.84, 0.975])
    return dict(mean=float(x.mean()), std=float(x.std()), median=float(q[2]),
                q16=float(q[1]), q84=float(q[3]),
                q025=float(q[0]), q975=float(q[4]), n=int(x.size))

def _ess_single_chain(x):
    # Cheap ESS via autocorrelation truncated at first non-positive lag.
    x = np.asarray(x, dtype=float)
    n = x.size
    y = x - x.mean()
    var0 = float((y * y).mean())
    if var0 == 0.0 or n < 4:
        return float(n)
    max_lag = min(n - 1, 200)
    acf = np.array([float((y[: n - k] * y[k:]).mean() / var0)
                    for k in range(1, max_lag + 1)])
    cut_idx = np.where(acf <= 0.0)[0]
    end = int(cut_idx[0]) if cut_idx.size else len(acf)
    tau = 1.0 + 2.0 * float(acf[:end].sum())
    return float(n / max(tau, 1.0))

def _ext_summary(ex):
    s = {}
    if not ex:
        return s
    if "diverging" in ex:
        d = np.asarray(ex["diverging"])
        s["n_diverging"] = int(d.sum())
        s["frac_diverging"] = float(d.mean())
    if "num_steps" in ex:
        ns = np.asarray(ex["num_steps"])
        s["num_steps"] = dict(mean=float(ns.mean()), median=float(np.median(ns)),
                              max=int(ns.max()), p99=float(np.quantile(ns, 0.99)))
    if "accept_prob" in ex:
        ap = np.asarray(ex["accept_prob"])
        s["accept_prob"] = dict(mean=float(ap.mean()), min=float(ap.min()),
                                frac_lt_0p1=float((ap < 0.1).mean()))
    if "energy" in ex:
        s["energy_std"] = float(np.asarray(ex["energy"]).std())
    return s

per_truth = {}
posteriors_npz = {}
vmr_npz = {}
spectra_npz = {
    "nu_obs": np.asarray(nu_obs),
    "pressure_bar": np.asarray(art.pressure),
    "species_labels": np.array(species_labels),
}
nuts_npz = {}

for T in TRUTHS:
    name = T["_name"]
    truth_vec = np.array([T[p] for p in PARAMS])
    co_truth = float(10.0 ** (T["log_C_H"] - T["log_O_H"]))

    # VMR agreement at truth
    Tarr_t = art.powerlaw_temperature(T["T0"], T["alpha"])
    abund_t = jnp.array([T[p] for p in ABUND_PARAMS])
    vc = np.asarray(vmr_classical(Tarr_t, abund_t))          # (NLAYER, 17)
    vm = np.asarray(vmr_emulator(Tarr_t, abund_t))
    eps = 1e-300
    lr = np.log10(np.clip(vm, eps, None)) - np.log10(np.clip(vc, eps, None))
    per_species = {
        sp: dict(
            max_abs_dex=float(np.nanmax(np.abs(lr[:, i]))),
            median_abs_dex=float(np.nanmedian(np.abs(lr[:, i]))),
            rms_dex=float(np.sqrt(np.nanmean(lr[:, i] ** 2))),
        )
        for i, sp in enumerate(species_labels)
    }
    vmr_npz[f"{name}/classical"] = vc
    vmr_npz[f"{name}/emulator"] = vm
    vmr_npz[f"{name}/log10_ratio_ml_over_cl"] = lr
    vmr_npz[f"{name}/Tarr"] = np.asarray(Tarr_t)

    # Spectrum residuals at truth (noiseless) and MAP reconstructions
    mu_cl_truth = np.asarray(fspec_classical_jit(**_kw(T)))
    mu_ml_truth = np.asarray(fspec_emulator_jit(**_kw(T)))
    res_truth = mu_ml_truth - mu_cl_truth
    mu_cl_map = np.asarray(fspec_classical_jit(*MAPS[name]["classical"]["x"]))
    mu_ml_map = np.asarray(fspec_emulator_jit(*MAPS[name]["emulator"]["x"]))
    spectra_npz[f"{name}/Fobs"] = MOCKS[name]["Fobs"]
    spectra_npz[f"{name}/mu_classical_at_truth"] = mu_cl_truth
    spectra_npz[f"{name}/mu_emulator_at_truth"] = mu_ml_truth
    spectra_npz[f"{name}/mu_classical_at_map"] = mu_cl_map
    spectra_npz[f"{name}/mu_emulator_at_map"] = mu_ml_map
    spec_resid = dict(
        rms=float(np.sqrt(np.mean(res_truth ** 2))),
        max_abs=float(np.max(np.abs(res_truth))),
    )
    spec_resid["rms_over_noise"] = spec_resid["rms"] / NOISE

    # NLL at truth and at each MAP (cross-evaluated).
    nll_cl_fn, _ = NLLS[name]["classical"]
    nll_ml_fn, _ = NLLS[name]["emulator"]
    x_truth = jnp.asarray(truth_vec)
    x_cl = jnp.asarray(MAPS[name]["classical"]["x"])
    x_ml = jnp.asarray(MAPS[name]["emulator"]["x"])
    nll_eval = dict(
        cl_at_truth=float(nll_cl_fn(x_truth)),
        ml_at_truth=float(nll_ml_fn(x_truth)),
        cl_at_cl_map=float(nll_cl_fn(x_cl)),
        ml_at_ml_map=float(nll_ml_fn(x_ml)),
        cl_at_ml_map=float(nll_cl_fn(x_ml)),
        ml_at_cl_map=float(nll_ml_fn(x_cl)),
    )

    # Posterior stats (+ derived C/O)
    sc = MCMC_CL[name]["samples"]
    sm = MCMC_ML[name]["samples"]
    co_cl = 10.0 ** (np.asarray(sc["log_C_H"]) - np.asarray(sc["log_O_H"]))
    co_ml = 10.0 ** (np.asarray(sm["log_C_H"]) - np.asarray(sm["log_O_H"]))

    stats_cl, stats_ml = {}, {}
    for p in PARAMS:
        xs_cl = np.asarray(sc[p])
        xs_ml = np.asarray(sm[p])
        stats_cl[p] = {
            **_pstats(xs_cl),
            "ess": _ess_single_chain(xs_cl),
            "bias_vs_truth": float(xs_cl.mean() - T[p]),
            "z_vs_truth": float((xs_cl.mean() - T[p]) / max(xs_cl.std(), 1e-12)),
        }
        stats_ml[p] = {
            **_pstats(xs_ml),
            "ess": _ess_single_chain(xs_ml),
            "bias_vs_truth": float(xs_ml.mean() - T[p]),
            "z_vs_truth": float((xs_ml.mean() - T[p]) / max(xs_ml.std(), 1e-12)),
        }
    stats_cl["C/O"] = {**_pstats(co_cl),
                       "bias_vs_truth": float(co_cl.mean() - co_truth)}
    stats_ml["C/O"] = {**_pstats(co_ml),
                       "bias_vs_truth": float(co_ml.mean() - co_truth)}

    width_ratio = {p: float(stats_ml[p]["std"] / max(stats_cl[p]["std"], 1e-12))
                   for p in PARAMS}
    width_ratio["C/O"] = float(
        stats_ml["C/O"]["std"] / max(stats_cl["C/O"]["std"], 1e-12)
    )

    # NUTS extras
    ex_cl = MCMC_CL[name]["mcmc"].get_extra_fields()
    ex_ml = MCMC_ML[name]["mcmc"].get_extra_fields()
    for k, v in ex_cl.items():
        nuts_npz[f"{name}/classical/{k}"] = np.asarray(v)
    for k, v in ex_ml.items():
        nuts_npz[f"{name}/emulator/{k}"] = np.asarray(v)

    per_truth[name] = {
        "truth": {**{p: float(T[p]) for p in PARAMS}, "C/O": co_truth},
        "map": {
            "classical": {
                **{p: float(MAPS[name]["classical"]["x"][i]) for i, p in enumerate(PARAMS)},
                "nll_star": MAPS[name]["classical"]["nll"],
                "nfev": MAPS[name]["classical"]["nfev"],
                "njev": MAPS[name]["classical"]["njev"],
                "wall_s": MAPS[name]["classical"]["wall_s"],
                "success": MAPS[name]["classical"]["success"],
            },
            "emulator": {
                **{p: float(MAPS[name]["emulator"]["x"][i]) for i, p in enumerate(PARAMS)},
                "nll_star": MAPS[name]["emulator"]["nll"],
                "nfev": MAPS[name]["emulator"]["nfev"],
                "njev": MAPS[name]["emulator"]["njev"],
                "wall_s": MAPS[name]["emulator"]["wall_s"],
                "success": MAPS[name]["emulator"]["success"],
            },
        },
        "posterior_stats": {"classical": stats_cl, "emulator": stats_ml},
        "posterior_width_ratio_ml_over_cl": width_ratio,
        "nll": nll_eval,
        "spectrum_residual_ml_minus_cl_at_truth": spec_resid,
        "vmr_log10_ratio_ml_over_cl_at_truth": per_species,
        "nuts_extras_summary": {
            "classical": _ext_summary(ex_cl),
            "emulator": _ext_summary(ex_ml),
        },
        "mcmc_wall_s": {
            "classical": float(MCMC_CL[name]["wall_s"]),
            "emulator": float(MCMC_ML[name]["wall_s"]),
        },
    }

    for k, v in sc.items():
        posteriors_npz[f"{name}/classical/{k}"] = np.asarray(v)
    for k, v in sm.items():
        posteriors_npz[f"{name}/emulator/{k}"] = np.asarray(v)
    posteriors_npz[f"{name}/truth_vec"] = truth_vec
    posteriors_npz[f"{name}/C_over_O_classical"] = co_cl
    posteriors_npz[f"{name}/C_over_O_emulator"] = co_ml

summary = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "bundle_path": str(BUNDLE_PATH.resolve()),
    "chemistry_type": str(bundle.chemistry_type),
    "model_type": str(bundle.model_type),
    "species_labels": list(species_labels),
    "params": list(PARAMS),
    "nlayer": int(NLAYER),
    "noise": float(NOISE),
    "abund_prior_sigma_dex": float(ABUND_PRIOR_SIGMA),
    "nuts": {
        "num_warmup": int(NUM_WARMUP),
        "num_samples": int(NUM_SAMPLES),
        "max_tree_depth": int(MAX_TREE),
    },
    "timings_ms_per_call": dict(FSPEC_TIMING),
    "per_truth": per_truth,
}

(OUTDIR / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
np.savez_compressed(OUTDIR / "posteriors.npz", **posteriors_npz)
np.savez_compressed(OUTDIR / "vmr_at_truth.npz", **vmr_npz)
np.savez_compressed(OUTDIR / "spectra.npz", **spectra_npz)
if nuts_npz:
    np.savez_compressed(OUTDIR / "nuts_extras.npz", **nuts_npz)
else:
    print("note: NUTS extras empty — re-run the MCMC cells after the Cell 21 edit "
          "to capture diverging/num_steps/accept_prob diagnostics.")

print("wrote:")
for f in sorted(OUTDIR.iterdir()):
    print(f"  {f.name:<22} {f.stat().st_size/1e3:7.1f} KB")

# Headline stdout: the thing to eyeball first.
print()
print("HEADLINE — bias = MCMC posterior mean - truth  (dex for log_X_H; absolute otherwise)")
for name, pt in per_truth.items():
    print(f"[{name}]   (truth C/O = {pt['truth']['C/O']:.3f})")
    for p in PARAMS:
        bc = pt["posterior_stats"]["classical"][p]["bias_vs_truth"]
        bm = pt["posterior_stats"]["emulator"][p]["bias_vs_truth"]
        wr = pt["posterior_width_ratio_ml_over_cl"][p]
        print(f"  {p:<10}  cl {bc:+7.3f}   ml {bm:+7.3f}   width(ml/cl) {wr:5.2f}")
    bc = pt["posterior_stats"]["classical"]["C/O"]["bias_vs_truth"]
    bm = pt["posterior_stats"]["emulator"]["C/O"]["bias_vs_truth"]
    wr = pt["posterior_width_ratio_ml_over_cl"]["C/O"]
    print(f"  {'C/O':<10}  cl {bc:+7.3f}   ml {bm:+7.3f}   width(ml/cl) {wr:5.2f}")
    sp = pt["spectrum_residual_ml_minus_cl_at_truth"]
    print(f"  spec RMS(ml-cl @ truth) = {sp['rms']:.2f}  "
          f"({sp['rms_over_noise']:.2f} × noise,  max |resid| = {sp['max_abs']:.2f})")
    ne = pt["nuts_extras_summary"].get("emulator", {})
    if "n_diverging" in ne:
        print(f"  NUTS (emulator): diverging={ne['n_diverging']}  "
              f"accept_prob mean={ne['accept_prob']['mean']:.2f}  "
              f"num_steps mean={ne['num_steps']['mean']:.1f} p99={ne['num_steps']['p99']:.0f}")
